<a href="https://colab.research.google.com/github/JakobDuffin/BATC-Data/blob/main/daylio_jake.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('daylio_jd.csv')

In [ ]:
df.head(10)

,full_date,date,weekday,time,mood,activities,note_title,note
0,2025-12-09,Dec 9,Tuesday,11:33 PM,meh,happy | grateful | content | angry | Engaged |...,NaN,Spent all day getting both echelon machines ru...
1,2025-12-08,Dec 8,Monday,11:59 PM,meh,angry | desperate | Frustrated | Distracted | ...,NaN,Spent all morning working on ordering stuff an...
2,2025-12-07,Dec 7,Sunday,11:59 PM,good,happy | anxious | Engaged | In love ❤️ | Fun |...,NaN,I woke up slow with Maranda. She came on to me...
3,2025-12-06,Dec 6,Saturday,9:30 PM,rad,happy | grateful | relaxed | content | In love...,NaN,"Went down to the Butt Christmas party, it was ..."
4,2025-12-05,Dec 5,Friday,9:30 PM,good,happy | excited | relaxed | tired | anxious | ...,NaN,Worked on the FA analysis today. Got interrupt...
5,2025-12-04,Dec 4,Thursday,11:07 PM,meh,angry | stressed | Engaged | Frustrated | Focu...,NaN,Heidi blew up on me on the nine o’clock. I was...
6,2025-12-03,Dec 3,Wednesday,9:30 PM,good,tired | anxious | Engaged | Distracted | Horny...,NaN,Hunters back at work. I started on the FA file...
7,2025-12-02,Dec 2,Tuesday,11:08 PM,meh,tired | angry | Frustrated | Distracted | Weak...,NaN,I did a bunch of Nicknacky stuff in the mornin...
8,2025-12-01,Dec 1,Monday,9:30 PM,meh,tired | anxious | stressed | Engaged | Fun | F...,NaN,Spent all day overhauling the kanban catalog a...
9,2025-11-30,Nov 30,Sunday,8:55 PM,good,happy | grateful | relaxed | content | Engaged...,NaN,Woke up sluggish to Maranda telling me it was ...


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 384 entries, 0 to 383
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   full_date   384 non-null    object 
 1   date        384 non-null    object 
 2   weekday     384 non-null    object 
 3   time        384 non-null    object 
 4   mood        384 non-null    object 
 5   activities  382 non-null    object 
 6   note_title  0 non-null      float64
 7   note        384 non-null    object 
dtypes: float64(1), object(7)
memory usage: 24.1+ KB


In [ ]:
# Raw csv file cleaning (run only once after connecting raw file)

     # date conversion and extraction
df['full_date'] = pd.to_datetime(df['full_date'])
df['year'] = df['full_date'].dt.year
df['month'] = df['full_date'].dt.month
df['day'] = df['full_date'].dt.day

     # Dropping unneccissary columns
df=df.drop(columns=['full_date'])
df=df.drop(columns=['date'])
df=df.drop(columns=['note_title'])

     # Rename column(s)
df = df.rename(columns={'time': 'entry_time'})
df = df.rename(columns={'weekday': 'day_name'})

     # activities to list conversion
import re
df['activities'] = df['activities'].str.split(r'\s*\|\s*')


     # Mood string name to numeric score conversion/col addition
df['mood'] = df['mood'].str.strip().str.lower()
mood_order = ['awful', 'bad', 'meh', 'good', 'rad']
mood_to_score = {m: i+1 for i, m in enumerate(mood_order)}   # awful=1 … best=5
df['m_score'] = df['mood'].map(mood_to_score)

     # Column index re-ordering
df = df[['year', 'month', 'day', 'day_name', 'entry_time', 'mood', 'm_score', 'activities', 'note']]


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1YkyDM9dUi1zOKOQVSzNos63-qtuMFyAxvr4tamvMvvA/edit#gid=0


In [ ]:
# Mood subset creation

df[['mood' , 'm_score']].head(20)

bad = df[df['mood']=='bad']
bad.head()

good = df[df['mood']=='good']
good.head()

rad = df[df['mood']=='rad']
rad.head()

meh = df[df['mood']=='meh']
meh.head()

,year,month,day,day_name,entry_time,mood,m_score,activities,note
0,2025,12,9,Tuesday,11:33 PM,meh,3,"[happy, grateful, content, angry, Engaged, Fru...",Spent all day getting both echelon machines ru...
1,2025,12,8,Monday,11:59 PM,meh,3,"[angry, desperate, Frustrated, Distracted, Foc...",Spent all morning working on ordering stuff an...
5,2025,12,4,Thursday,11:07 PM,meh,3,"[angry, stressed, Engaged, Frustrated, Focused...",Heidi blew up on me on the nine o’clock. I was...
7,2025,12,2,Tuesday,11:08 PM,meh,3,"[tired, angry, Frustrated, Distracted, Weak, B...",I did a bunch of Nicknacky stuff in the mornin...
8,2025,12,1,Monday,9:30 PM,meh,3,"[tired, anxious, stressed, Engaged, Fun, Focus...",Spent all day overhauling the kanban catalog a...


In [ ]:
# Activity extraction

activity_counts = (
    df['activities']
    .explode()
    .value_counts()
)


In [ ]:
# Print activities per day annual percentages

print(round((activity_counts/360)*100,1))

activities
Vaped               98.6
homemade            84.4
happy               80.3
Brushed             73.9
sunny               73.1
                    ... 
Dirt Bike            0.3
Racquetball          0.3
end on time          0.3
Snoozed              0.3
X Country skiing     0.3
Name: count, Length: 130, dtype: float64
